In [1]:
# Célula 1 — Imports
import geopandas as gpd
import pandas as pd
from pathlib import Path

In [2]:
# Célula 2 — Carregar dados
pasta = Path(r"C:\Users\franc\OneDrive\Francisco\Profissional\MBA_Data_Science_ e_Analytics\00_TCC\06_Dados_base\GEO\2024_02_basegeo")

lf = gpd.read_file(pasta / "lf.shp")
bf = gpd.read_file(pasta / "bf.shp")

lf_val = lf[lf.geometry.notna() & lf.geometry.is_valid].copy()
bf_val = bf[bf.geometry.notna() & bf.geometry.is_valid].copy()

In [3]:
# Célula 3 — 5 pares de SOBREPOSIÇÃO REAL entre NUMBLOCO cadastrados (≠ 000000000000)
LIMIAR_BORDA = 5.0
LIMIAR_REAL  = 50.0

joined = gpd.sjoin(lf_val, bf_val, how="inner", predicate="overlaps")

resultados = []
for idx, row in joined.iterrows():
    nb_lf = lf_val.loc[idx, "NUMBLOCO"]
    idx_b = row["index_right"]
    nb_bf = bf_val.loc[idx_b, "NUMBLOCO"]

    # Somente pares onde AMBOS têm NUMBLOCO cadastrado
    if nb_lf == "000000000000" or nb_bf == "000000000000":
        continue
    if nb_lf is None or nb_bf is None:
        continue

    geom_lf = lf_val.loc[idx, "geometry"]
    geom_bf = bf_val.loc[idx_b, "geometry"]

    if not geom_lf.intersects(geom_bf):
        continue

    intersec  = geom_lf.intersection(geom_bf)
    area_over = intersec.area
    area_lf   = geom_lf.area
    area_bf   = geom_bf.area
    pct_lf    = area_over / area_lf * 100 if area_lf > 0 else 0
    pct_bf    = area_over / area_bf * 100 if area_bf > 0 else 0

    if area_over < LIMIAR_REAL:
        continue

    # Excluir o par já conhecido
    if {nb_lf, nb_bf} == {"007027410000", "007027420000"}:
        continue

    resultados.append({
        "NUMBLOCO_LF":  nb_lf,
        "AREA_LF":      round(area_lf, 1),
        "NUMBLOCO_BF":  nb_bf,
        "AREA_BF":      round(area_bf, 1),
        "AREA_OVERLAP": round(area_over, 1),
        "% LF":         round(pct_lf, 1),
        "% BF":         round(pct_bf, 1),
    })

df = pd.DataFrame(resultados).drop_duplicates(subset=["NUMBLOCO_LF", "NUMBLOCO_BF"])
print("5 pares — SOBREPOSIÇÃO REAL entre NUMBLOCO cadastrados (≠ 000000000000)\n")
print(df.head(5).to_string(index=False))

5 pares — SOBREPOSIÇÃO REAL entre NUMBLOCO cadastrados (≠ 000000000000)

 NUMBLOCO_LF  AREA_LF  NUMBLOCO_BF  AREA_BF  AREA_OVERLAP  % LF  % BF
001133960000    918.0 001133970000    860.6          51.8   5.6   6.0
001019070000    576.8 000064360000    376.2          59.6  10.3  15.8
002430590000   2865.9 001758050000   1527.6         267.9   9.3  17.5
002039570000  17724.1 000060060000  19587.9         146.4   0.8   0.7
001921610000   3069.4 005007670007   2408.4          51.0   1.7   2.1
